# Stage 2 — Agent pipeline

How the second stage works, end to end, before any LLM is plugged in. The code lives in `agent/`; this notebook shows it in use.

Run order: `kyc.ipynb` → `preprocessing.ipynb` → `xgboost.ipynb` → this notebook.

This notebook covers:
1. One flagged case, as a reviewer sees it
2. What each tool returns for that case
3. The pipeline with the no-LLM baseline reviewer
4. Plugging in an LLM

**Leakage rules, enforced in code and tested in `agent/test_tools.py`** (`python -m pytest agent -q`):
- Tools never load `is_fraud`, and the simulator's `fraud_` prefix is stripped from merchant names.
- Each case's tools are tied to that customer and that moment: they take no customer or date arguments, and return only strictly earlier transactions.
- Labels live only in `scores_{split}.csv`, which the evaluation reads; reviewers never see them.

In [ ]:
import json

from agent.data import TransactionStore
from agent.pipeline import evaluate, load_flagged, load_thresholds, run
from agent.prompt import SYSTEM_PROMPT, render_case
from agent.reviewers import ScoreThresholdReviewer
from agent.schemas import Case
from agent.tools import TOOL_DEFINITIONS, CaseTools

store = TransactionStore.load()
thresholds = load_thresholds()
flagged = load_flagged("val")
print(f"Thresholds: standalone {thresholds['standalone']:.3f}, flagging {thresholds['flag']:.3f} "
      f"(catches {thresholds['recall_target']:.0%} of validation fraud)")
print(f"Flagged validation cases: {len(flagged):,}")

## 1. One flagged case

This is everything a reviewer is given up front: no label and no card number. The customer is identified only by their KYC record ID.

In [ ]:
row = flagged.sort_values("xgb_score").iloc[len(flagged) // 2]  # a mid-scoring case, not an obvious one
case = Case(
    case_id=row.trans_num,
    customer_id=store.customer_id(row.cc_num),
    time=row.trans_date_trans_time,
    amount=row.amt,
    merchant=row.merchant,
    category=row.category,
    xgb_score=row.xgb_score,
)
tools = CaseTools(store, row.cc_num, row.trans_date_trans_time)
print(render_case(case))

## 2. What each tool returns

These are the exact outputs an LLM would receive for this case.

In [ ]:
print(json.dumps(tools.call("get_kyc_profile"), indent=2))

In [ ]:
print(json.dumps(tools.call("get_transaction_history", {"limit": 10}), indent=2))

In [ ]:
print(json.dumps(tools.call("get_spending_summary"), indent=2))

## 3. Pipeline run with the baseline reviewer

`ScoreThresholdReviewer` confirms a flag when the XGBoost score clears the standalone threshold, so **XGBoost + baseline reviewer must reproduce XGBoost alone exactly**. That checks the plumbing (case loading, tools, run log, scaling of sampled runs) before an LLM is involved.

It runs on a random sample of 500 flagged validation cases, the way the LLM runs will. Runs are written to `data/agent_runs/` and can be resumed.

In [ ]:
reviewer = ScoreThresholdReviewer(thresholds["standalone"])
run_path = run("val", reviewer, sample=500, run_name="demo-baseline", store=store)
evaluate("val", run_path).round(3)

Rows A (sample, scaled) and B should match each other exactly, and be close to A (exact). The gap between them is sampling error: a 500-case sample estimates the full flagged set. The **Stage 1 flags** row is the ceiling: frauds below the flagging threshold never reach the reviewer, so no reviewer can catch them.

## 4. The LLM reviewer

`agent/llm.py` runs the agent loop over any OpenAI-compatible API (OpenAI, Gemini, OpenRouter, local Ollama), so the provider and model are settings. It needs `pip install openai` and the provider's API key. Start with a 20-case pilot to measure tokens and cost per case:

```python
from agent.llm import HISTORY_TOOLS, LLMReviewer, OpenAICompatibleBackend

backend = OpenAICompatibleBackend("openrouter", "<model name>")
run_path = run("val", LLMReviewer(backend), sample=20, store=store)          # system B
# LLMReviewer(backend, include_score=False)                                 # system C: score withheld
# LLMReviewer(backend, allowed_tools=HISTORY_TOOLS)                         # system D: no KYC tool
evaluate("val", run_path)
```

This is the prompt and the tools every LLM reviewer gets:

In [ ]:
print(SYSTEM_PROMPT)
print()
print([d["name"] for d in TOOL_DEFINITIONS])